# Fine-tuning of a pretrained foundation model
Using the ESOL dataset, we have seen that the linear-probe approach of a foundation model with a small regressor on top is about comparable with small regression models trained on Morgan fingerprints, but performs worse than the small models, when they are trained on molecular descriptors.

Fine-tuning makes the model more adaptive towards the target task. In this exercise, you will use basically the same architecture (`ChemBERTa-zinc-base-v1` + regressor (this time with one hidden layer)), but instead of using the pretrained model as a fixed encoder only, it is (partially) fine-tuned on the ESOL data. 

Since fine-tuning takes quite a bit of time, the notebook has been prerun and the final state of the model was saved for another transfer learning task (7B_TransferLearning).

## Your task in this notebook: 
Go through the code and try to understand the essential sections. Imagine now you are building a model for your team and you want your colleagues to understand the code as well - hence, comments are essentials (also as a reminder for yourself). **Write short comments in lines with 3 hashtags (###)** (sections 2-9). The aim is to **very briefly(!)** explain the syntax or used features / parameters, etc to elucidate the function of that line. 


1) Import dependencies

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np


2) Prepare the data: Import the dataset, train-test split, define the dataset class

In [18]:
df_full = pd.read_csv("esol.csv")


df_full.dropna(axis=0, inplace=True) ### drop rows with missing values


# df = df_full
df = df_full.sample(100, random_state=42) # use either the full dataset or a much smaller one by sampling
print(df.head())
print(f"Dataset size: {len(df)}")

                                   smiles   logS
1091                             CC/C=C\C -2.540
898        O=C1NC(=O)NC(=O)C1(CC)CC=C(C)C -2.253
739      Cc1[nH]c(=O)n(c(=O)c1Cl)C(C)(C)C -2.484
140                              CC/C=C/C -2.540
1019  ClC(Cl)C(c1ccc(Cl)cc1)c2ccc(Cl)cc2  -7.200
Dataset size: 100


In [19]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42 ###  train 80% test 20% and random state is for rerpoducability so that if 
    # someone else runs the code don't get different splits
)

In [20]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128): ### initilaization of the class ESOLdataset which has always
        # a df and tokenizer and default maxmimum length set to 128
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        label = self.df.iloc[idx]["logS"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length", ### if the sequence is shorter than the max length, pad it with zeros
            max_length=self.max_length,
            return_tensors="pt" ### return the encodings as PyTorch tensors
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0), ### return the input ids as a 1D tensor instead of a 2D tensor with batch size 1
            # squeeze(0) removes the extra dimension added by return_tensors="pt" which makes the shape (1, max_length) and we want it to be (max_length,)
            "attention_mask": enc["attention_mask"].squeeze(0), ### return the attention mask as a 1D tensor instead of a 2D tensor with batch size 1
            "labels": torch.tensor(label, dtype=torch.float)
        }


3) Define the architecture:

In [21]:
class chemberta_esol_regressor(nn.Module):
    def __init__(self, model_name, hidden_dim=None):
        super().__init__()

        # general encoder
        self.encoder = AutoModel.from_pretrained(model_name) ### use the pretrained ChemBERTa model as the encoder for the regression task

        if hidden_dim is None:
            hidden_dim = self.encoder.config.hidden_size ### if hidden_dim is not provided, use the hidden size of the encoder as the hidden dimension for the regression head

        # Regression head (task-specific)
        self.fc1 = nn.Linear(hidden_dim, 256) ###  first fully connected layer takes chemical features and reduces them to 256 numbers
        self.act = nn.ReLU() ### Activation function: turns negative numbers to zero (adds non-linearity)
        self.dropout = nn.Dropout(p=0.2) 
        self.fc2 = nn.Linear(256, 1) ###  Final layer: reduces 256 numbers down to 1 number (the solubility prediction)

    def forward(self, input_ids, attention_mask): ### forward method defines how the input data flows through the model to produce the output
        # Encoder forward
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Pooling (explicit & visible)
        # Use [CLS] token representation
        cls_embedding = outputs.last_hidden_state[:, 0, :] 
        ### Takes the [CLS] token (special first token) which contains the whole molecule's summary

        # Regression head forward
        x = self.fc1(cls_embedding)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x.squeeze(-1)


4) Load foundation model

In [22]:
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1" ### pretrained ChemBERTa model trained
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = chemberta_esol_regressor(MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


5) Define model parameters

In [23]:
BATCH_SIZE = 32 ### define the batch size for training and validation dataloaders
EPOCHS = 20

###  learning rates for the encoder and the regression head
LR_ENCODER = 1e-5
LR_HEAD = 1e-3

N_UNFROZEN_LAYERS = 2  ###  number of layers to unfreeze during fine-tuning

6) Partial fine-tuning

6.1. Freeze everything:

In [24]:
for param in model.encoder.parameters(): # "encoder" here is the model name defined in our NN architecture
    param.requires_grad = False ### freeze all layers so that their weights are not updated during training since 
    # the model is already pretrained and I don't want to mess them up since they are already good. which I believe is the whole point of using a pretrained model in the first place.
    


6.2. Unfreeze last k transformer layers of foundation model

In [25]:
encoder_layers = model.encoder.encoder.layer # the first "encoder" is the name defined in our model architecture,
# the second one is the location, where the layers are stored in the ChemBERTa model, e.g. model.chemberta.encoder.layer

for layer in encoder_layers[-N_UNFROZEN_LAYERS:]: ### we want now to unfreeze some layers so that we adapt some weights to our specific task and dataset.
    # so we unfreeze the last N_UNFROZEN_LAYERS layers of the encoder

        param.requires_grad = True ### unfreeze the parameters of the last N_UNFROZEN_LAYERS layers so that they can be updated during training and adapt to our specific task and dataset

6.3. Optionally: Unfreeze / retrain final Layer Normalisation of the encoder

In [26]:
for param in model.encoder.embeddings.LayerNorm.parameters(): # "encoder" here is the model name defined in our NN architecture
    param.requires_grad = True


7) Initiate DataLoaders

In [27]:
train_dataset = ESOLDataset(train_df, tokenizer)
val_dataset   = ESOLDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) ### since the data set is very big, we need to use a dataloader to load the data in batches during 
# training and shuffle the training data to make sure the model doesn't learn the order of the data
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE) ### the same goes for validation data but we don't need to shuffle it since we are not training on it


8) Define Optimizer with different learning rates for the encoder and the regressor model

In [28]:
optimizer = AdamW(
    [
        {
            "params": encoder_layers[-N_UNFROZEN_LAYERS:].parameters(), ### only the last N_UNFROZEN_LAYERS layers of the encoder will be updated during training
            "lr": LR_ENCODER,
        },
        {
            "params": list(model.fc1.parameters()) + list(model.fc2.parameters()),
            "lr": LR_HEAD,
        },
    ],
    weight_decay=1e-2 ### weight decay for regularization
)

loss_fn = nn.MSELoss()



9) Define evaluation function and initiate training.

In [29]:
def evaluate(model, dataloader):
    model.eval() ### evaluation function to evaluate the model on the validation set.
    preds = []  
    targets = []
    with torch.no_grad(): ### 
        for batch in dataloader:
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"] ### Tells model which tokens are real (1) vs padding (0), ignore the zeros remember from padding!
            labels = batch["labels"]

            outputs = model(input_ids, attention_mask)
            preds.append(outputs.cpu().numpy())
            targets.append(labels.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    return rmse

In [33]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward() ### backpropagation to compute the gradients of the loss with respect to the model parameters
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() ###  accumulate the loss for the epoch to compute the average loss at the end of the epoch

    train_loss = total_loss / len(train_loader)
    val_rmse = evaluate(model, val_loader)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train RMSE: {np.sqrt(train_loss):.4f} | "
        f"Val RMSE: {val_rmse:.4f}"
    )

Epoch 01 | Train RMSE: 0.8556 | Val RMSE: 1.2635
Epoch 02 | Train RMSE: 0.8714 | Val RMSE: 1.3435
Epoch 03 | Train RMSE: 0.8051 | Val RMSE: 1.3542
Epoch 04 | Train RMSE: 0.8770 | Val RMSE: 1.3827
Epoch 05 | Train RMSE: 1.1179 | Val RMSE: 1.3317
Epoch 06 | Train RMSE: 0.8425 | Val RMSE: 1.2078
Epoch 07 | Train RMSE: 0.7815 | Val RMSE: 1.3036
Epoch 08 | Train RMSE: 0.7997 | Val RMSE: 1.5153
Epoch 09 | Train RMSE: 0.8810 | Val RMSE: 1.3204
Epoch 10 | Train RMSE: 0.8153 | Val RMSE: 1.2947
Epoch 11 | Train RMSE: 0.8252 | Val RMSE: 1.4851
Epoch 12 | Train RMSE: 0.7563 | Val RMSE: 1.6764
Epoch 13 | Train RMSE: 0.8364 | Val RMSE: 1.3241
Epoch 14 | Train RMSE: 0.8648 | Val RMSE: 1.2293
Epoch 15 | Train RMSE: 0.8306 | Val RMSE: 1.3259
Epoch 16 | Train RMSE: 0.7592 | Val RMSE: 1.3956
Epoch 17 | Train RMSE: 0.7479 | Val RMSE: 1.3906
Epoch 18 | Train RMSE: 0.7270 | Val RMSE: 1.3618
Epoch 19 | Train RMSE: 0.6651 | Val RMSE: 1.3696
Epoch 20 | Train RMSE: 0.7787 | Val RMSE: 1.5096


10) Save the task-specific encoder as pretrained model - Hugging Face (HF) style (this can be reloaded like any HF model - see snippet in assignment 7B). Note: This is not the same as saving the entire model via pytorch (`torch.save(model)`) - the solution below is ergo not unstable and brittle, but highly reusable for encodings!

In [31]:
model.encoder.save_pretrained("chemberta_esol_encoder")
tokenizer.save_pretrained("chemberta_esol_encoder")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('chemberta_esol_encoder/tokenizer_config.json',
 'chemberta_esol_encoder/tokenizer.json')

11) Save the state of the regressor-head to reload the pretrained regressor later on (could be done also for the entire model (including the encoder), however, the filesize would be larger).

In [32]:
# save states for entire model
# torch.save(model.state_dict(), "chemberta_esol_regressor.pt")

torch.save({
    "fc1": model.fc1.state_dict(),
    "fc2": model.fc2.state_dict(),
}, "chemberta_esol_regressor_head.pt")

Note that the model architecture is not stored in any of these state dicts - either move it to a separate file (e.g. `models.py`), or copy-paste the class definition wherever you need it again - simply strip it down to the bare architecture (see 7B).